# Optional secondary robustness metrics
    These tools reuse public benchmark data schemas. Their local inference protocol
    is labeled modified. No paid scoring API is called. ACCreason requires a correct
    first error **and** a correct explanation: it remains null until you supply
    provenance-bearing human/official-judge decisions. Merely identifying a step
    is not reported as ACCreason.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

candidates = [Path(os.environ.get("GASP_ROOT", ".")), Path.cwd(), Path.cwd().parent, Path("/content/gasp")]
ROOT = next((p.resolve() for p in candidates if (p / "src/gasp").is_dir()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Unzip dist/gasp-colab.zip and open this notebook inside the project.")
    print("Select the delivered gasp-colab.zip archive.")
    uploaded = files.upload()
    archives = [p for p in uploaded if p.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Select exactly one project archive")
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            if not name.startswith("gasp/") or ".." in Path(name).parts:
                raise ValueError("Unexpected archive member")
        z.extractall("/content")
    ROOT = Path("/content/gasp")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

def command(*args):
    result = subprocess.run([sys.executable, *map(str,args)], cwd=ROOT,
                            env={**os.environ,"PYTHONPATH":str(ROOT/"src")},
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()

print("Project:", ROOT)


In [ ]:
# Optional persistence. Keep downloaded HF weights in Colab scratch; persist data/runs.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/gasp-experiment")
else:
    WORK = ROOT
WORK.mkdir(parents=True, exist_ok=True)
print("Data and outputs:", WORK)


In [ ]:
RUN = WORK/"runs/full-v4/split-42"
FETCH_PUBLIC_BENCHMARKS = False
if FETCH_PUBLIC_BENCHMARKS:
    for name,url in [("MR-GSM8K","https://github.com/JIA-Lab-research/MR-GSM8K"),
                     ("Mr-Ben","https://github.com/JIA-Lab-research/Mr-Ben"),
                     ("RUPBench","https://github.com/EternityYW/RUPBench")]:
        dest=WORK/"references"/name
        if not dest.exists():subprocess.run(["git","clone","--depth","1",url,str(dest)],check=True)
        print(name,subprocess.check_output(["git","-C",str(dest),"rev-parse","HEAD"],text=True).strip())


Use the exact upstream file you inspected; record its git commit. File names
    can change. The adapters fail on unexpected schemas instead of guessing labels.
    For MHPP choose only the coding subset, not the entire Mr-Ben collection.


In [ ]:
# Set paths and enable only after obtaining the real files.
PREPARE_SECONDARY = False
if PREPARE_SECONDARY:
    from gasp.benchmarks import prepare_mr, prepare_rup
    mr_repo=WORK/"references/MR-GSM8K"
    revision=subprocess.check_output(["git","-C",str(mr_repo),"rev-parse","HEAD"],text=True).strip()
    prepare_mr(mr_repo/"dataset/MR-GSM8K.json",WORK/"data/secondary/mr.jsonl",
               "mr-gsm8k","https://github.com/JIA-Lab-research/MR-GSM8K",revision)
    # prepare_mr(mhpp_file, ..., "mhpp", mr_ben_url, exact_commit)
    # prepare_rup(gsm_csv, ..., "gsm8k", rupbench_url, exact_commit)
    # prepare_rup(cqa_csv, ..., "cqa", rupbench_url, exact_commit)


In [ ]:
RUN_SECONDARY = False
if RUN_SECONDARY:
    command("-m","gasp.benchmarks","evaluate","--run",RUN,
            "--cache",WORK/"data/secondary/mr.jsonl","--output",RUN/"secondary/mr")
    command("-m","gasp.benchmarks","self-revision","--run",RUN,
            "--cache",WORK/"data/full-v4/eval.jsonl","--output",RUN/"secondary/revision")


The diagnostic output includes `judge_template.json`. Supply a descriptive
    `judge_provenance` and binary `explanation_correct` decisions, retaining prediction
    hashes, then score with the command below. Do this for both conditions using the
    same judging process. MHPP initial-clean-solved filtering needs a coding verifier
    and is not provided; report full-set diagnosis separately. Self-revision uses each
    agent's own failed set as a secondary metric, not the primary sharing comparison.


In [ ]:
SCORE_EXPLANATIONS = False
if SCORE_EXPLANATIONS:
    command("-m","gasp.benchmarks","score-diagnosis",
            "--predictions",RUN/"secondary/mr/predictions.jsonl",
            "--judge",RUN/"secondary/mr/judge-completed.json",
            "--output",RUN/"secondary/mr/reason-score.json")
